# Cross-Modal Diagnostic Observability — Stage T3-PF

## Outcome-free preregistration freeze, executable-edge resolution and asset preflight v1.0

这一本只做**下一阶段真正允许做的事情**：

- 验证 Stage T2-F 的 12/12 formal record；
- 冻结 Selective Diagnostic Observability v0.6、RA-CB-AMW-DDET v0.3 和 Stage T3 preregistration v1.0；
- 解析原始 13 条 locked-blind candidate edges 与已冻结 source axes 的交集；
- 生成 T3-A sentinel / T3-B expansion 的最终治理记录；
- 检查标签防火墙、资产获取条件和下一册授权边界。

**不会下载、解压、解析或评分任何 locked-blind target，也不会读取任何 blind outcome。**

干净 Colab CPU 运行时，直接 `Run all`。末尾只有出现  
`SEAL_T3_PREREG_AUTHORISE_OUTCOME_FREE_ASSET_ACQUISITION_ONLY`  
才允许进入受控资产获取册。

In [1]:
# @title T3-PF-0. Mount Drive, verify Stage T2-F and freeze the final theory/method/preregistration
import hashlib, json, os, platform, sys
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display=print

IN_COLAB=False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB=True
except Exception:
    pass

DEFAULT_ROOT=Path('/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability') if IN_COLAB else Path.cwd()
PROJECT_ROOT=Path(os.environ.get('CDO_PROJECT_ROOT',str(DEFAULT_ROOT)))
CODE_ROOT=PROJECT_ROOT/'05_Code'/'Cross_Modal'
CM_ROOT=PROJECT_ROOT/'06_Data_Records'/'Cross_Modal'
RESULT_ROOT=CM_ROOT/'StageT3-PF_Outcome-Free_Preregistration_And_Asset_Preflight_v1.0'
P0,P1,P2,P3,P4=[RESULT_ROOT/x for x in ['00_Protocol','01_Executable_Edges','02_Asset_Preflight','03_Firewall','04_Results']]
for p in [CODE_ROOT,P0,P1,P2,P3,P4]: p.mkdir(parents=True,exist_ok=True)

NOTEBOOK_NAME='CrossModal_StageT3-PF_Outcome-Free_Preregistration_And_Asset_Preflight_v1.0.ipynb'
NOTEBOOK_PATH=CODE_ROOT/NOTEBOOK_NAME
THEORY_NAME='Directed_Diagnostic_Evidence_Transport_Selective_Observability_Theory_v0.6.md'
METHOD_NAME='RA-CB-AMW-DDET_Method_Specification_v0.3.md'
PREREG_NAME='StageT3_RA-CB-AMW-DDET_Prospective_Validation_Preregistration_v1.0.md'

THEORY_PATH=PROJECT_ROOT/'03_Theory'/'Directed_Diagnostic_Evidence_Transport_v0.6'/THEORY_NAME
METHOD_PATH=PROJECT_ROOT/'03_Theory'/'Directed_Diagnostic_Evidence_Transport_v0.6'/METHOD_NAME
PREREG_PATH=PROJECT_ROOT/'04_Study_Design'/PREREG_NAME
T2F_FINAL=CM_ROOT/'StageT2-F_Development_Only_RA-CB-AMW-DDET_Risk_Adaptive_Covariate_Balance_And_Blind_Refreeze_v0.1'/'04_Results'/'StageT2-F_Complete_v0.1.json'
STAGE10=CM_ROOT/'Stage10_Preassigned_Development_And_Locked_Blind_Registry_v0.1'
LOCKED_REGISTRY=STAGE10/'04_Locked_Blind_Reserve'/'Stage10_Frozen_Locked_Blind_Target_Edge_Registry_v0.1.csv'
LOCKED_AUDIT=STAGE10/'04_Locked_Blind_Reserve'/'Stage10_Locked_Blind_Reserve_Audit_v0.1.csv'
RIGHTS_LEDGER=STAGE10/'02_Endpoint_And_Access_Audit'/'Stage10_Official_Source_Rights_And_Access_Ledger_v0.1.csv'
ENDPOINT_RULES=STAGE10/'02_Endpoint_And_Access_Audit'/'Stage10_Frozen_Label_Mapping_And_Grouping_Audit_v0.1.csv'
QUARANTINE_CONTRACT=STAGE10/'04_Locked_Blind_Reserve'/'Stage10_Frozen_Locked_Blind_Label_Quarantine_Contract_v0.1.json'

STAGE8=CM_ROOT/'Stage8_CrossModality_EdgeLibrary_Expansion_v0.1'
DERM_SOURCE_MANIFEST=STAGE8/'03_Frozen_Source_Axes'/'Stage8_Source_Recoverability_Summary_v0.1.csv'
STAGE11E=CM_ROOT/'Stage11E-R_Development_Only_Source_Recoverability_And_Axis_Freeze_v0.1'
US_SOURCE_MANIFEST=STAGE11E/'03_Frozen_Source_Axes'/'Stage11E-R_Frozen_Source_Axis_Manifest_v0.1.csv'

EXPECTED_FILE_SHA={
    't2f_final':'5fa777822cbe28540f8f1b78ea0c96c4dbc4d6213212ce61558f6d662bca19c4',
    'locked_registry':'98e37f294ece94cd5b86d91db209828101d9bd42074f1ba2c91d4f72760cea40',
    'locked_audit':'f798073134ace2a7b943113ebbb3f18df163a98f9b5627fabb1071ad15f05903',
    'theory':'bb4675972475091ed7da9adbef1b4ca22deea7ba62967d1fc56c5422a57f327b',
    'method':'9f59961d5486cb22a551e1c92d57e3322a7a7a31ee4d6eb37c3d19977a1360e2',
    'preregistration':'e01a0e4d2e82215ad746ddd03d29ef8a1467eea7e6872418fb3ac66b7d564c7f',
}
EXPECTED_T2F_FINAL_RECORD='a7dc616c2cf46c772bcd452a7f5804c77f67eae553214217f7525ce59ea6c7e9'
LOCKED_TARGETS=['BUSI_CAIRO_2019','OASBUD_2017','DERM7PT_2019']

def now(): return datetime.now(timezone.utc).isoformat()
def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def sha_json(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
def canonical_csv(frame):
    return frame.fillna('').to_csv(index=False,lineterminator='\n',float_format='%.12g')
def write_once(path,text):
    path=Path(path)
    if path.exists():
        assert path.read_text(encoding='utf-8')==text,f'Replay conflict: {path}'
    else:
        path.write_text(text,encoding='utf-8')
def write_csv_once(path,frame): write_once(path,canonical_csv(frame))
def write_json_once(path,value): write_once(path,json.dumps(value,indent=2,ensure_ascii=False)+'\n')
def verify_self(path,field):
    value=json.loads(Path(path).read_text(encoding='utf-8'))
    claim=value[field]; core=dict(value); core.pop(field)
    assert sha_json(core)==claim,f'Self-hash mismatch: {path}'
    return value
def notebook_source_sha(path):
    value=json.loads(Path(path).read_text(encoding='utf-8')); cells=[]
    for cell in value.get('cells',[]):
        if cell.get('cell_type') not in {'code','markdown'}: continue
        source=cell.get('source',[]); source=''.join(source) if isinstance(source,list) else str(source)
        cells.append({'cell_type':cell['cell_type'],'source':source.replace('\r\n','\n')})
    return sha_json(cells)
def seal(path,payload,hash_field='seal_sha256'):
    path=Path(path)
    if path.exists():
        old=verify_self(path,hash_field)
        for k,v in payload.items(): assert old[k]==v,f'Sealed field changed: {k}'
        return old
    value=dict(payload); value['sealed_utc']=now(); value[hash_field]=sha_json(value)
    write_json_once(path,value); return value

required=[NOTEBOOK_PATH,THEORY_PATH,METHOD_PATH,PREREG_PATH,T2F_FINAL,LOCKED_REGISTRY,LOCKED_AUDIT,
          RIGHTS_LEDGER,ENDPOINT_RULES,QUARANTINE_CONTRACT,DERM_SOURCE_MANIFEST,US_SOURCE_MANIFEST]
missing=[str(p) for p in required if not p.is_file()]
assert not missing,'Missing immutable files:\n'+'\n'.join(missing)
for role,path in {
    't2f_final':T2F_FINAL,'locked_registry':LOCKED_REGISTRY,'locked_audit':LOCKED_AUDIT,
    'theory':THEORY_PATH,'method':METHOD_PATH,'preregistration':PREREG_PATH}.items():
    assert sha_file(path)==EXPECTED_FILE_SHA[role],f'Hash mismatch: {role}'

t2f=verify_self(T2F_FINAL,'final_record_sha256')
assert t2f['final_record_sha256']==EXPECTED_T2F_FINAL_RECORD
assert t2f['frozen_gates_passed']==t2f['frozen_gates_total']==12
assert t2f['decision']=='AUTHORISE_RA_CB_AMW_DDET_V0_2_FREEZE_AND_T3_V0_2_PREREGISTRATION_ONLY'
assert t2f['stage12_authorised'] is False and t2f['locked_blind_assets_touched'] is False

protocol_payload={
    'stage':'StageT3-PF',
    'purpose':'outcome_free_final_preregistration_and_asset_preflight',
    'parent_t2f_final_record_sha256':EXPECTED_T2F_FINAL_RECORD,
    'theory_sha256':EXPECTED_FILE_SHA['theory'],
    'method_sha256':EXPECTED_FILE_SHA['method'],
    'preregistration_sha256':EXPECTED_FILE_SHA['preregistration'],
    'locked_registry_sha256':EXPECTED_FILE_SHA['locked_registry'],
    'locked_audit_sha256':EXPECTED_FILE_SHA['locked_audit'],
    'notebook_source_sha256':notebook_source_sha(NOTEBOOK_PATH),
    'blind_outcomes_accessed':False,
    'blind_assets_acquired':False,
    'stage12_authorised':False,
}
protocol=seal(P0/'StageT3-PF_Protocol_Seal_v1.0.json',protocol_payload)
print('T2-F parent:',EXPECTED_T2F_FINAL_RECORD)
print('T3 preregistration SHA256:',EXPECTED_FILE_SHA['preregistration'])
print('T3-PF protocol seal:',protocol['seal_sha256'])
print('Blind outcomes accessed: False')


Mounted at /content/drive
T2-F parent: a7dc616c2cf46c772bcd452a7f5804c77f67eae553214217f7525ce59ea6c7e9
T3 preregistration SHA256: e01a0e4d2e82215ad746ddd03d29ef8a1467eea7e6872418fb3ac66b7d564c7f
T3-PF protocol seal: fe9fa8e6629130347660e9e1ab24805faf61ca3111fb7c9da26ebfb7d8a35b82
Blind outcomes accessed: False


In [2]:
# @title T3-PF-1. Resolve candidate edges against the genuinely frozen source axes
registry=pd.read_csv(LOCKED_REGISTRY)
audit=pd.read_csv(LOCKED_AUDIT)
rights=pd.read_csv(RIGHTS_LEDGER)
endpoint=pd.read_csv(ENDPOINT_RULES)
derm=pd.read_csv(DERM_SOURCE_MANIFEST)
us=pd.read_csv(US_SOURCE_MANIFEST)

assert len(registry)==13 and registry.target.nunique()==3
assert set(registry.target)==set(LOCKED_TARGETS)
assert not registry.target_labels_accessed.astype(bool).any()
assert not registry.target_eligible_for_fit.astype(bool).any()
assert audit.set_index('gate').loc['blind_labels_accessed_in_stage10','observed']==0

source_alias={'MSK-1':'ISIC_MSK1','UDA-1':'ISIC_UDA1'}
recoverable=set(derm.loc[derm.recoverable.astype(bool),'source'].astype(str))
recoverable.update(us.dataset_id.astype(str))
registry['resolved_source_id']=registry.source.map(source_alias).fillna(registry.source)
registry['source_axis_frozen']=registry.resolved_source_id.isin(recoverable)
registry['executable_pre_asset']=registry.source_axis_frozen
registry['pre_asset_status']=np.where(registry.executable_pre_asset,
    'ELIGIBLE_PENDING_TARGET_ASSET_AND_GROUPING_PREFLIGHT',
    'BLOCKED_FROZEN_SOURCE_AXIS_UNAVAILABLE')
registry['outcome_accessed']=False

# Preserve every preassigned edge; do not silently delete the two BREAST_LESIONS_USG rows.
eligible=registry[registry.executable_pre_asset].copy()
blocked=registry[~registry.executable_pre_asset].copy()
assert len(eligible)==11 and len(blocked)==2
assert set(blocked.source)=={'BREAST_LESIONS_USG_2024'}

target_summary=registry.groupby(['target','modality','task'],as_index=False).agg(
    candidate_edges=('edge_id','size'),
    executable_edges_pre_asset=('executable_pre_asset','sum'),
    blocked_source_edges=('executable_pre_asset',lambda x:(~x).sum()),
    target_labels_accessed=('target_labels_accessed','max'))
target_summary['validation_tier']='T3-A_SENTINEL'
target_summary['confirmatory_generalisation_claim']=False

write_csv_once(P1/'StageT3-PF_All_Preassigned_Edges_With_Resolution_v1.0.csv',registry)
write_csv_once(P1/'StageT3-PF_Executable_Sentinel_Edge_Registry_v1.0.csv',eligible)
write_csv_once(P1/'StageT3-PF_Blocked_Edge_Ledger_v1.0.csv',blocked)
write_csv_once(P1/'StageT3-PF_Sentinel_Target_Summary_v1.0.csv',target_summary)

display(target_summary)
display(blocked[['edge_id','source','target','pre_asset_status']])
print('Executable pre-asset edges:',len(eligible),'of',len(registry))
print('All blocked rows remain in the immutable ledger.')


,target,modality,task,candidate_edges,executable_edges_pre_asset,blocked_source_edges,target_labels_accessed,validation_tier,confirmatory_generalisation_claim
0,BUSI_CAIRO_2019,breast_ultrasound,breast_lesion_malignant_vs_benign,5,4,1,False,T3-A_SENTINEL,False
1,DERM7PT_2019,dermoscopy,melanoma_vs_melanocytic_nevus,3,3,0,False,T3-A_SENTINEL,False
2,OASBUD_2017,breast_ultrasound,breast_lesion_malignant_vs_benign,5,4,1,False,T3-A_SENTINEL,False


,edge_id,source,target,pre_asset_status
0,BLIND::BREAST_LESIONS_USG_2024__TO__BUSI_CAIRO...,BREAST_LESIONS_USG_2024,BUSI_CAIRO_2019,BLOCKED_FROZEN_SOURCE_AXIS_UNAVAILABLE
1,BLIND::BREAST_LESIONS_USG_2024__TO__OASBUD_2017,BREAST_LESIONS_USG_2024,OASBUD_2017,BLOCKED_FROZEN_SOURCE_AXIS_UNAVAILABLE


Executable pre-asset edges: 11 of 13
All blocked rows remain in the immutable ledger.


In [3]:
# @title T3-PF-2. Freeze asset rules, T3-A/T3-B separation and expansion candidate ledger
blind_rights=rights[rights.dataset_id.isin(LOCKED_TARGETS)].copy()
assert len(blind_rights)==3
assert blind_rights.stage10_data_status.eq('NOT_ACQUIRED').all()

asset_rules=pd.DataFrame([
    {'target':'BUSI_CAIRO_2019','tier':'T3-A_SENTINEL','official_route':'Cairo University author release',
     'label_coupling':'class directory/filename coupled','required_grouping':'patient; lesion nested if available',
     'preflight_action':'opaque extraction; prove patient grouping or mark UNAVAILABLE',
     'outcome_access_allowed':False},
    {'target':'OASBUD_2017','tier':'T3-A_SENTINEL','official_route':'Zenodo record 545928 v1',
     'label_coupling':'RF and label metadata bundled in MAT','required_grouping':'patient/lesion from official container',
     'preflight_action':'presealed RF renderer plus opaque label quarantine',
     'outcome_access_allowed':False},
    {'target':'DERM7PT_2019','tier':'T3-A_SENTINEL','official_route':'SFU Derm7pt author release',
     'label_coupling':'metadata separate but access controlled','required_grouping':'lesion or patient where available',
     'preflight_action':'dermoscopy-only image acquisition; metadata quarantine',
     'outcome_access_allowed':False},
])
write_csv_once(P2/'StageT3-PF_Sentinel_Asset_And_Quarantine_Rules_v1.0.csv',asset_rules)

# Candidate-only list. No target is promoted by this notebook.
expansion=pd.DataFrame([
    {'candidate':'PH2','modality':'dermoscopy','task':'melanoma_vs_melanocytic_nevus','status':'CANDIDATE_ONLY',
     'required_before_freeze':'official version/license; lesion IDs; overlap audit; frozen source-axis compatibility'},
    {'candidate':'DDR','modality':'retinal_fundus','task':'referable_DR','status':'CANDIDATE_ONLY',
     'required_before_freeze':'official release/license; patient IDs; endpoint mapping; source-axis compatibility'},
    {'candidate':'MESSIDOR2','modality':'retinal_fundus','task':'referable_DR','status':'CANDIDATE_ONLY',
     'required_before_freeze':'exact label provenance; exam grouping; derivative overlap audit'},
    {'candidate':'FGADR','modality':'retinal_fundus','task':'referable_DR','status':'CANDIDATE_ONLY',
     'required_before_freeze':'signed research agreement; patient IDs; official manifest'},
    {'candidate':'INDEPENDENT_TB_CXR_TARGET','modality':'chest_radiography','task':'tuberculosis_vs_normal','status':'CANDIDATE_ONLY',
     'required_before_freeze':'official public target not used in development; patient grouping; source-axis compatibility'},
    {'candidate':'INDEPENDENT_CLINICAL_SKIN_TARGET','modality':'clinical_skin_photography','task':'melanoma_vs_nevus_or_frozen_binary_endpoint','status':'CANDIDATE_ONLY',
     'required_before_freeze':'unseen-modality endpoint compatibility; patient IDs; sufficient groups; prospective roster seal'},
])
expansion['outcome_accessed']=False
expansion['counts_toward_confirmatory_n']=False
write_csv_once(P2/'StageT3-PF_T3B_Expansion_Candidate_Ledger_NOT_YET_FROZEN_v1.0.csv',expansion)

tier_rules={
    'T3-A':{'targets':3,'modalities':2,'role':'prospective_sentinel_kill_test',
            'confirmatory_generalisation':False,'edges_pre_asset':int(len(eligible))},
    'T3-B':{'minimum_targets':6,'minimum_modalities':3,'prefer_unseen_modality':True,
            'role':'confirmatory_replication','currently_frozen_targets':0},
}
write_json_once(P2/'StageT3-PF_Two_Tier_Validation_Rules_v1.0.json',tier_rules)
display(asset_rules)
print('T3-B promoted targets: 0. Candidate outcomes remain untouched.')


,target,tier,official_route,label_coupling,required_grouping,preflight_action,outcome_access_allowed
0,BUSI_CAIRO_2019,T3-A_SENTINEL,Cairo University author release,class directory/filename coupled,patient; lesion nested if available,opaque extraction; prove patient grouping or m...,False
1,OASBUD_2017,T3-A_SENTINEL,Zenodo record 545928 v1,RF and label metadata bundled in MAT,patient/lesion from official container,presealed RF renderer plus opaque label quaran...,False
2,DERM7PT_2019,T3-A_SENTINEL,SFU Derm7pt author release,metadata separate but access controlled,lesion or patient where available,dermoscopy-only image acquisition; metadata qu...,False


T3-B promoted targets: 0. Candidate outcomes remain untouched.


In [4]:
# @title T3-PF-3. Run firewall gates and write the final preregistration activation record
# Executable inputs must contain only governance, source-axis and development result paths.
resolved_inputs='|'.join(map(str,[T2F_FINAL,LOCKED_REGISTRY,LOCKED_AUDIT,RIGHTS_LEDGER,ENDPOINT_RULES,
                                  QUARANTINE_CONTRACT,DERM_SOURCE_MANIFEST,US_SOURCE_MANIFEST,
                                  THEORY_PATH,METHOD_PATH,PREREG_PATH,NOTEBOOK_PATH]))
assert not any(('/'+target.lower()+'/') in resolved_inputs.lower() for target in LOCKED_TARGETS)

gates=[
    ('G1_t2f_formal_pass',t2f['frozen_gates_passed']==12,'12/12 formal development gates'),
    ('G2_t2f_no_blind_touch',t2f['locked_blind_assets_touched'] is False,'False required'),
    ('G3_final_documents_exact',
     sha_file(THEORY_PATH)==EXPECTED_FILE_SHA['theory'] and sha_file(METHOD_PATH)==EXPECTED_FILE_SHA['method'] and sha_file(PREREG_PATH)==EXPECTED_FILE_SHA['preregistration'],
     'theory/method/prereg hashes exact'),
    ('G4_stage10_registry_exact',sha_file(LOCKED_REGISTRY)==EXPECTED_FILE_SHA['locked_registry'],'registry hash exact'),
    ('G5_preassigned_targets_preserved',set(registry.target)==set(LOCKED_TARGETS) and len(registry)==13,'3 targets / 13 candidate edges'),
    ('G6_executable_sources_resolved',len(eligible)==11 and len(blocked)==2,'11 executable / 2 blocked ledger rows'),
    ('G7_label_flags_untouched',not registry.target_labels_accessed.astype(bool).any(),'all False'),
    ('G8_asset_status_unacquired',blind_rights.stage10_data_status.eq('NOT_ACQUIRED').all(),'all NOT_ACQUIRED'),
    ('G9_tier_separation',tier_rules['T3-A']['confirmatory_generalisation'] is False and tier_rules['T3-B']['currently_frozen_targets']==0,'sentinel != confirmatory expansion'),
    ('G10_no_outcomes_loaded',True,'this notebook imports no blind images, labels, archives or outcome tables'),
    ('G11_notebook_source_recorded',bool(protocol['notebook_source_sha256']),'source SHA present'),
    ('G12_stage12_still_false',protocol['stage12_authorised'] is False,'False required'),
]
gates=pd.DataFrame(gates,columns=['gate','passed','observed'])
assert gates.passed.all()
write_csv_once(P3/'StageT3-PF_Firewall_And_Activation_Gates_v1.0.csv',gates)

decision='SEAL_T3_PREREG_AUTHORISE_OUTCOME_FREE_ASSET_ACQUISITION_ONLY'
activation={
    'stage':'StageT3-PF',
    'decision':decision,
    'parent_t2f_final_record_sha256':EXPECTED_T2F_FINAL_RECORD,
    'protocol_seal_sha256':protocol['seal_sha256'],
    'theory_sha256':EXPECTED_FILE_SHA['theory'],
    'method_sha256':EXPECTED_FILE_SHA['method'],
    'preregistration_sha256':EXPECTED_FILE_SHA['preregistration'],
    'preassigned_targets':3,
    'preassigned_candidate_edges':13,
    'executable_edges_before_target_asset_preflight':11,
    'blocked_source_axis_edges_preserved':2,
    't3a_confirmatory_generalisation_claim':False,
    't3b_frozen_targets':0,
    'blind_assets_acquired':False,
    'blind_outcomes_accessed':False,
    'outcome_scoring_authorised':False,
    'stage12_authorised':False,
}
activation['activation_record_sha256']=sha_json(activation)
write_json_once(P4/'StageT3-PF_Activation_Record_v1.0.json',activation)

summary=f"""# Stage T3-PF result summary v1.0

- Decision: `{decision}`
- T2-F parent: `{EXPECTED_T2F_FINAL_RECORD}`
- T3 preregistration SHA256: `{EXPECTED_FILE_SHA['preregistration']}`
- T3-A sentinel targets: 3
- Preassigned candidate edges: 13
- Executable edges after frozen-source resolution: 11
- Blocked edges preserved in ledger: 2
- T3-B confirmatory targets currently frozen: 0
- Blind assets acquired: False
- Blind outcomes accessed: False
- Outcome scoring authorised: False
- Stage 12 authorised: False

This run freezes the final theory, method, two-tier validation logic and asset rules. It authorises only controlled outcome-free asset acquisition and target-specific grouping/renderer preflight. It does not authorise blind scoring.
"""
write_once(P4/'StageT3-PF_Result_Summary_v1.0.md',summary)

display(gates)
print('\\n========== STAGE T3-PF COMPLETE ==========')
print('Decision:',decision)
print('Executable sentinel edges:',len(eligible))
print('Blind assets acquired:',False)
print('Blind outcomes accessed:',False)
print('Outcome scoring authorised:',False)
print('Activation record SHA256:',activation['activation_record_sha256'])


,gate,passed,observed
0,G1_t2f_formal_pass,True,12/12 formal development gates
1,G2_t2f_no_blind_touch,True,False required
2,G3_final_documents_exact,True,theory/method/prereg hashes exact
3,G4_stage10_registry_exact,True,registry hash exact
4,G5_preassigned_targets_preserved,True,3 targets / 13 candidate edges
5,G6_executable_sources_resolved,True,11 executable / 2 blocked ledger rows
6,G7_label_flags_untouched,True,all False
7,G8_asset_status_unacquired,True,all NOT_ACQUIRED
8,G9_tier_separation,True,sentinel != confirmatory expansion
9,G10_no_outcomes_loaded,True,"this notebook imports no blind images, labels,..."


\n========== STAGE T3-PF COMPLETE ==========
Decision: SEAL_T3_PREREG_AUTHORISE_OUTCOME_FREE_ASSET_ACQUISITION_ONLY
Executable sentinel edges: 11
Blind assets acquired: False
Blind outcomes accessed: False
Outcome scoring authorised: False
Activation record SHA256: 4397cee7798f684159ed77aa5e1edd7b7ae0a24378047d6c89b37ef9ef738a52


## 运行结果怎么解释

- 这一本通过，表示理论、方法、预注册、原始 blind roster、可执行源轴和防火墙已经统一冻结。
- 原始 13 条 blind candidate edges 中，只有真正具有冻结 source axis 的 11 条进入后续资产预检；其余 2 条保留为 blocked ledger，不能静默删除。
- T3-A 的 3 个 targets 是严格 prospective sentinel，但数量不足以单独确认跨模态普适性。
- T3-B 必须另外 outcome-free 冻结不少于 6 个 targets、3 个 modalities。
- 本册不接触 blind data，因此末尾仍应显示 `Outcome scoring authorised: False`。这是正确结果，不是失败。